# Baseline: Tables 1 and 2

This notebook **fits the majority, linear and quadratic logistic models** from the supplied labelled arrays. Each fit uses the fixed seed-67 70/20/10 split, with train and validation combined (90%) as in the paper. The same local-test partition is scored only after fitting. Deterministic methods are fitted three times under the paper’s three seed labels.

In [1]:
NAME="baseline"
from pathlib import Path
from datetime import datetime, timezone
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "splits/manifest.json").exists())
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from IPython.display import display, Image
from common import load_split
SEEDS = (42, 123, 2026)
RUN = ROOT / "results/notebook_reruns" / (NAME + "-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
RUN.mkdir(parents=True)
print("Split seed 67; training seeds", SEEDS)
print("Fresh run directory:", RUN.relative_to(ROOT))

Split seed 67; training seeds (42, 123, 2026)
Fresh run directory: results/notebook_reruns/baseline-20261119T021052895955Z


## Table 1: shared partitions

In [2]:
import json
manifest=json.loads((ROOT/"splits/manifest.json").read_text())
assert manifest["seed"] == 67
sizes=[]
for d in ("20","30","40","40-adapt"):
    entry=manifest["datasets"][d]
    sizes.append({"Dataset":"K"+d.replace("-adapt","-Adapt"),"Train":entry["sizes"]["train"],"Validation":entry["sizes"]["validation"],"Local test":entry["sizes"]["test"]})
display(pd.DataFrame(sizes).set_index("Dataset"))

,Train,Validation,Local test
Dataset,,,
K20,8400,2400,1200
K30,14000,4000,2000
K40,22400,6400,3200
K40-Adapt,2800,800,400


## Fit on 90% train + validation

The fixed logistic settings are defined in `baseline/train.py`; no local-test labels enter this cell.

In [3]:
from baseline.train import build_model
from threadpoolctl import threadpool_limits
import warnings
from sklearn.exceptions import ConvergenceWarning
methods=("majority","linear-logistic","quadratic-logistic")
models={}
with threadpool_limits(limits=1):
    for d in ("20","30","40"):
        x_train,y_train,x_val,y_val,_,_=load_split(d)
        x_fit=np.concatenate((x_train,x_val))
        y_fit=np.concatenate((y_train,y_val))
        for method in methods:
            for seed in SEEDS:
                model=build_model(method)
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore",ConvergenceWarning)
                    model.fit(x_fit,y_fit)
                models[d,method,seed]=model
                print(f"Fitted K{d} {method} seed {seed}: {len(y_fit)} rows")

Fitted K20 majority seed 42: 10800 rows
Fitted K20 majority seed 123: 10800 rows
Fitted K20 majority seed 2026: 10800 rows
Fitted K20 linear-logistic seed 42: 10800 rows
Fitted K20 linear-logistic seed 123: 10800 rows
Fitted K20 linear-logistic seed 2026: 10800 rows
Fitted K20 quadratic-logistic seed 42: 10800 rows
Fitted K20 quadratic-logistic seed 123: 10800 rows
Fitted K20 quadratic-logistic seed 2026: 10800 rows
Fitted K30 majority seed 42: 18000 rows
Fitted K30 majority seed 123: 18000 rows
Fitted K30 majority seed 2026: 18000 rows
Fitted K30 linear-logistic seed 42: 18000 rows
Fitted K30 linear-logistic seed 123: 18000 rows


Fitted K30 linear-logistic seed 2026: 18000 rows


Fitted K30 quadratic-logistic seed 42: 18000 rows


Fitted K30 quadratic-logistic seed 123: 18000 rows


Fitted K30 quadratic-logistic seed 2026: 18000 rows
Fitted K40 majority seed 42: 28800 rows
Fitted K40 majority seed 123: 28800 rows
Fitted K40 majority seed 2026: 28800 rows
Fitted K40 linear-logistic seed 42: 28800 rows
Fitted K40 linear-logistic seed 123: 28800 rows
Fitted K40 linear-logistic seed 2026: 28800 rows


Fitted K40 quadratic-logistic seed 42: 28800 rows
Fitted K40 quadratic-logistic seed 123: 28800 rows


Fitted K40 quadratic-logistic seed 2026: 28800 rows


## Evaluate the fixed 10% local test and calculate Table 2

In [4]:
rows=[]
for d in ("20","30","40"):
    _,_,_,_,x_test,y_test=load_split(d)
    for method in methods:
        for seed in SEEDS:
            predictions=models[d,method,seed].predict(x_test)
            rows.append({"dataset":d,"method":method,"seed":seed,"correct":int(np.sum(predictions==y_test)),"test_size":len(y_test),"accuracy_pct":100*float(np.mean(predictions==y_test))})
runs=pd.DataFrame(rows)
runs.to_csv(RUN/"local_test_runs.csv",index=False)
summary=runs.groupby(["dataset","method"])["accuracy_pct"].agg(mean="mean",sd="std")
names={"majority":"Majority","linear-logistic":"Linear logistic","quadratic-logistic":"Quadratic logistic"}
table=[]
for method,label in names.items():
    row={"Method":label}
    for d in ("20","30","40"):
        result=summary.loc[(d,method)]
        row[f"K{d}"]=f'{result["mean"]:.2f} ± {result["sd"]:.2f}'
    table.append(row)
display(pd.DataFrame(table).set_index("Method"))
print("Saved per-seed results to",(RUN/"local_test_runs.csv").relative_to(ROOT))

,K20,K30,K40
Method,,,
Majority,50.42 ± 0.00,50.05 ± 0.00,50.19 ± 0.00
Linear logistic,49.58 ± 0.00,48.95 ± 0.00,48.84 ± 0.00
Quadratic logistic,53.92 ± 0.00,54.50 ± 0.00,53.41 ± 0.00


Saved per-seed results to results/notebook_reruns/baseline-20261119T021052895955Z/local_test_runs.csv
